# COCO 2017 数据集深度分析

## 📋 项目简介

本 Notebook 对 **Microsoft COCO 2017** 目标检测数据集进行全方位深度分析。COCO (Common Objects in Context) 是目标检测领域最权威的基准数据集之一，包含 80 个类别、超过 11 万张训练图像和 86 万个标注实例。

### 分析维度

1. **类别分布与长尾分析** — 类别频率、Gini 系数、Head/Tail 划分
2. **边界框统计分析** — 尺寸分类、宽高比、空间位置热力图
3. **目标共现网络** — 共现矩阵、Lift 值、社区检测
4. **检测难度量化** — 小目标比例、密集度、综合难度评分
5. **数据增强策略建议** — 基于分析结果的具体增强方案

### 技术栈

`pandas` · `numpy` · `matplotlib` · `seaborn` · `plotly` · `pycocotools` · `networkx`

---

## 0. 环境准备

In [ ]:
import sys
from pathlib import Path

# 将项目根目录加入 Python path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.coco_analysis.data.loader import load_coco, coco_to_dataframe, get_dataset_summary
from src.coco_analysis.data.preprocessor import pipeline as preprocess_pipeline
from src.coco_analysis.analysis.category import generate_category_summary
from src.coco_analysis.analysis.bbox import generate_bbox_summary
from src.coco_analysis.analysis.cooccurrence import generate_cooccurrence_summary
from src.coco_analysis.analysis.difficulty import generate_difficulty_summary
from src.coco_analysis.analysis.augmentation import (
    recommend_mosaic_augmentation,
    recommend_class_balanced_sampling,
    summarize_recommendations,
)
from src.coco_analysis.visualization.theme import initialize_visualization
from src.coco_analysis.config import get_config

# 初始化可视化（中文字体 + 样式）
font_name = initialize_visualization()

# 加载配置
config = get_config()

print(f'✅ 环境初始化完成，使用字体: {font_name}')
print(f'   Python: {sys.version}')
print(f'   pandas: {pd.__version__}')
print(f'   numpy: {np.__version__}')

---

## 1. 数据加载与概览

加载 COCO 2017 标注文件并展示数据集的基本统计信息。

⚠️ **注意**: 如果标注文件不存在，请先运行 `python scripts/download_coco.py` 下载。

In [ ]:
# 配置标注文件路径
ANNOTATION_PATH = "data/coco/annotations/instances_train2017.json"

# 如果使用验证集（数据量更小，适合快速测试）
# ANNOTATION_PATH = "data/coco/annotations/instances_val2017.json"

print(f"📂 加载标注文件: {ANNOTATION_PATH}")

# 加载 COCO 数据
coco = load_coco(ANNOTATION_PATH)

# 转换为 DataFrame
df_raw = coco_to_dataframe(coco)

# 数据预处理
df = preprocess_pipeline(df_raw, config=config.analysis)

# 数据集概览
summary = get_dataset_summary(coco)
print(f"\n📊 COCO {config.coco.get('year', 2017)} 数据集概览")
print(f"{'='*40}")
print(f"  图像总数:     {summary['num_images']:>8,}")
print(f"  标注总数:     {summary['num_annotations']:>8,}")
print(f"  类别总数:     {summary['num_categories']:>8}")
print(f"  超类别数:     {len(summary['supercategories']):>8}")
print(f"  每图均值:     {summary['num_annotations']/summary['num_images']:>8.1f} 个目标")

# 展示 DataFrame 结构
print(f"\nDataFrame 形状: {df.shape}")
df.head()

---

## 2. 类别分布与长尾分析

分析 80 个目标类别的频率分布,量化数据集的类别不平衡程度。

**关键指标**:
- **Gini 系数**: 衡量分布均匀性 (0=完全均匀, 1=完全不均)
- **Head/Tail 划分**: 基于累积占比的类别分组

In [ ]:
from src.coco_analysis.visualization.category_plots import (
    plot_category_distribution,
    plot_long_tail_curve,
    plot_category_coverage,
)

# 执行类别分析
cat_results = generate_category_summary(df, config=config.analysis)

# 关键指标
gini = cat_results['gini_coefficient']
ht = cat_results['head_tail']

print(f"📊 类别分布关键指标")
print(f"{'='*40}")
print(f"  Gini 系数:         {gini:.3f}")
print(f"  Head 类别:         {ht['head_count']} 个 (占 {ht['head_total_pct']:.1f}% 实例)")
print(f"  Body 类别:         {ht['body_count']} 个")
print(f"  Tail 类别:         {ht['tail_count']} 个 (占 {ht['tail_total_pct']:.1f}% 实例)")
print()

# Top-5 / Bottom-5
print("🏆 Top-5 高频类别:")
for item in cat_results['top5_categories']:
    print(f"  {item['category_name']:20s}: {item['count']:>8,} ({item['percentage']:.1f}%)")

print()
print("🔻 Bottom-5 低频类别:")
for item in cat_results['bottom5_categories']:
    print(f"  {item['category_name']:20s}: {item['count']:>8,} ({item['percentage']:.2f}%)")

# 图表展示
fig1_path = plot_category_distribution(cat_results['category_frequency'])
fig2_path = plot_long_tail_curve(
    cat_results['category_frequency'],
    gini=gini,
    head_ratio=config.analysis['long_tail']['head_ratio'],
    tail_ratio=config.analysis['long_tail']['tail_ratio'],
)

### 超类别分布

COCO 将 80 个类别分为 12 个超类别 (person, vehicle, animal, food, etc.)

In [ ]:
supercat_df = cat_results['supercategory_distribution']

fig, ax = plt.subplots(figsize=(12, 6))

colors = plt.cm.Set3(np.linspace(0, 1, len(supercat_df)))
bars = ax.bar(supercat_df['supercategory'], supercat_df['instance_count'], color=colors, edgecolor='white')

for bar, count, pct, n_cats in zip(
    bars, supercat_df['instance_count'], supercat_df['instance_pct'], supercat_df['category_count']
):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(supercat_df['instance_count'])*0.01,
            f'{count:,}\n({pct:.1f}% / {n_cats}类)', ha='center', fontsize=8)

ax.set_xlabel('超类别', fontsize=12)
ax.set_ylabel('实例数量', fontsize=12)
ax.set_title('COCO 2017 超类别分布', fontsize=14, fontweight='bold')
ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(0, supercat_df['instance_count'].max() * 1.2)

plt.tight_layout()
plt.show()

---

## 3. 边界框统计分析

分析目标的尺寸、宽高比和空间分布特征。

**COCO 尺寸标准**:
- **Small**: 面积 ≤ 32² = 1024 px²
- **Medium**: 1024 < 面积 ≤ 96² = 9216 px²
- **Large**: 面积 > 96² px²

In [ ]:
from src.coco_analysis.visualization.bbox_plots import (
    plot_bbox_size_pie,
    plot_aspect_ratio_distribution,
    plot_position_heatmap,
    plot_bbox_area_violin,
)

# 执行 bbox 分析
bbox_results = generate_bbox_summary(df, config=config.analysis)

# 尺寸分布
size_cls = bbox_results['size_classification']
print(f"📊 目标尺寸分布")
print(f"{'='*40}")
for _, row in size_cls.iterrows():
    print(f"  {row['bbox_size_class']:10s}: {int(row['count']):>8,} ({row['percentage']:.1f}%)")

# 空间偏置
pos_bias = bbox_results['position_bias']
print(f"\n📍 空间位置偏置")
print(f"  中心 X 均值: {pos_bias['center_x_mean']:.3f}")
print(f"  中心 Y 均值: {pos_bias['center_y_mean']:.3f}")
print(f"  偏置方向: {pos_bias['center_bias_description']}")

# 图表
fig3_path = plot_bbox_size_pie(size_cls)
fig4_path = plot_aspect_ratio_distribution(df)
fig5_path = plot_position_heatmap(df)
fig6_path = plot_bbox_area_violin(df)

---

## 4. 目标共现网络分析

分析哪些类别的目标倾向于在同一张图像中一起出现。

**关键指标**:
- **共现矩阵**: 类别对共同出现的图像数
- **Lift 值**: 共现频率与随机期望的比值 (>1 = 正关联)
- **社区检测**: 通过 Louvain 算法发现语义分组

In [ ]:
from src.coco_analysis.visualization.network_plots import (
    plot_cooccurrence_heatmap,
    plot_cooccurrence_network_interactive,
    plot_top_cooccurring_chord,
)

# 执行共现分析
cooc_results = generate_cooccurrence_summary(df, config=config.analysis)

# 最频繁共现对
top_pairs = cooc_results['top_pairs']
print(f"🔗 Top 10 最频繁共现类别对:")
print(f"{'='*50}")
for pair in top_pairs[:10]:
    print(f"  {pair['cat1']:20s} ↔ {pair['cat2']:20s}: {pair['cooccurrence_count']:>6,} 张图像")

# 社区检测
communities = cooc_results.get('communities')
if communities:
    print(f"\n🌐 社区检测结果")
    print(f"  社区数量: {communities['num_communities']}")
    print(f"  模块度: {communities['modularity']:.3f}")
    for comm in communities['communities']:
        print(f"  社区 {comm['community_id']+1}: {', '.join(comm['members'])}")

# 热力图
fig7_path = plot_cooccurrence_heatmap(cooc_results['cooccurrence_matrix'])

# 交互式网络图 (需要 networkx)
cooc_matrix = cooc_results['cooccurrence_matrix']
lift_matrix = cooc_results['lift_matrix']

try:
    from src.coco_analysis.analysis.cooccurrence import build_cooccurrence_graph
    G = build_cooccurrence_graph(cooc_matrix, lift_matrix, min_edges=50, use_lift_weight=True)
    fig8_path = plot_cooccurrence_network_interactive(G, communities, output_dir='outputs/figures')
    print(f"\n  交互式网络图已保存: {fig8_path}")
except ImportError as e:
    print(f"\n  ⚠ 跳过交互式网络图: {e}")

---

## 5. 检测难度量化分析

从多个维度评估不同类别的检测难度：

| 维度 | 权重 | 说明 |
|------|------|------|
| 小目标比例 | 35% | 面积 ≤ 32² px 的目标占比 |
| 密集度 | 25% | 每张图像中的平均目标数 |
| 面积方差 | 20% | 同一图像内目标尺度的多样性 |
| 宽高比方差 | 10% | 目标形状的多样性 |
| Crowd 比例 | 10% | 密集人群标注的占比 |

In [ ]:
from src.coco_analysis.visualization.difficulty_plots import (
    plot_small_object_scatter,
    plot_objects_per_image_histogram,
    plot_difficulty_ranking,
)

# 执行难度分析
diff_results = generate_difficulty_summary(df, config=config.analysis)

# 最难/最易类别
print(f"🎯 检测难度排名")
print(f"{'='*50}")
print(f"\n🔴 最难检测 (Top 10):")
for cat in diff_results['hardest_categories']:
    print(f"  {cat['category_name']:20s}: 难度 {cat['difficulty_score']:5.1f} | {cat['total_count']:>6,} 实例")

print(f"\n🟢 最易检测 (Bottom 10):")
for cat in diff_results['easiest_categories']:
    print(f"  {cat['category_name']:20s}: 难度 {cat['difficulty_score']:5.1f} | {cat['total_count']:>6,} 实例")

# 图表
fig9_path = plot_small_object_scatter(df)
fig10_path = plot_objects_per_image_histogram(df)
fig11_path = plot_difficulty_ranking(diff_results['difficulty_by_category'])

---

## 6. 数据增强策略建议

基于以上分析结果，生成针对性的数据增强策略建议。

In [ ]:
# 生成增强策略
mosaic_recs = recommend_mosaic_augmentation(diff_results['difficulty_by_category'])
sampling_recs = recommend_class_balanced_sampling(
    cat_results['category_frequency'],
    cat_results['head_tail'],
)

# Mosaic 增强推荐
print(f"🖼️ Mosaic 增强推荐 (小目标占比 > 30%):")
print(f"{'='*50}")
if mosaic_recs:
    for rec in mosaic_recs[:10]:
        print(f"  [{rec['priority']:6s}] {rec['category_name']:20s} (小目标: {rec['small_ratio']:.1f}%)")
else:
    print(f"  无需特殊 Mosaic 增强")

# 过采样建议
oversample = sampling_recs.get('oversample_categories', [])
print(f"\n⚖️ Tail 类别过采样建议:")
print(f"{'='*50}")
for rec in oversample[:10]:
    print(f"  {rec['category']:20s}: {rec['current_count']:>6,} → {rec['target_count']:>6,} ({rec['suggested_oversample_ratio']:.1f}×)")

# 增强策略可视化
from src.coco_analysis.visualization.difficulty_plots import plot_augmentation_recommendations
fig12_path = plot_augmentation_recommendations(mosaic_recs, oversample)

# 文本摘要
aug_summary = summarize_recommendations(cat_results, diff_results, cooc_results)
from IPython.display import display, Markdown
display(Markdown(aug_summary))

---

## 7. 总结与展望

### 🔑 关键发现

通过本次对 COCO 2017 数据集的深度分析，我们从数据层面获得了以下洞察：

1. **类别分布高度不平衡** — Gini 系数通常在 0.4-0.6 之间，少数 Head 类别主导了大部分实例
2. **小目标检测是核心难点** — 小目标天然面积小、特征少，加之 COCO 中下采样导致的信息丢失
3. **目标共现遵循场景模式** — 社区检测能自动发现室内/室外、交通工具、食物等语义分组
4. **空间位置存在系统性偏置** — 目标可能倾向于出现在图像特定区域（如中心偏置）

### 💡 对目标检测模型的指导意义

- **模型架构**: 需要 FPN/PANet 等多尺度特征融合来应对尺度多样性
- **训练策略**: 多尺度训练 + Mosaic + MixUp 可有效提升小目标检测性能
- **损失函数**: 对长尾分布使用 Focal Loss 或 EQL (Equalization Loss)
- **数据采样**: 使用 RepeatFactorDataset 或 Class-aware Sampling 缓解不平衡

### 📚 延伸方向

- 将分析扩展到 COCO Keypoints (人体关键点) 和 Panoptic (全景分割)
- 对比 COCO 2014/2017 之间的分布偏移
- 将分析方法应用于自定义数据集
- 将分析结果集成到 MLOps 流程中（持续数据质量监控）

---

## 📎 附录: 报告导出

将分析结果导出为 HTML 报告，方便分享和展示。

In [ ]:
# 整合结果并导出完整报告
from src.coco_analysis.report.generator import generate_markdown_report, generate_html_report

# 整合所有结果
all_results = {
    'dataset_summary': summary,
    'category': cat_results,
    'bbox': bbox_results,
    'cooccurrence': cooc_results,
    'difficulty': diff_results,
    'augmentation_text': aug_summary,
}

# 收集图表路径
from pathlib import Path
figure_dir = Path('outputs/figures')
figure_paths = {}
for f in figure_dir.glob('*.png'):
    figure_paths[f.stem] = str(f)
for f in figure_dir.glob('*.html'):
    figure_paths[f.stem] = str(f)

# 生成报告
md_report = generate_markdown_report(all_results, figure_paths, lang='zh')
html_path = generate_html_report(md_report, figure_paths, 'outputs/reports/analysis_report.html')

print(f'✅ 报告导出完成!')
print(f'   HTML 报告: {html_path}')
print(f'   📂 所有图表: outputs/figures/')
print(f'\n💡 提示: 用浏览器打开 HTML 报告获得最佳阅读体验')